In [ ]:
import cv2
import json
import os
import numpy as np
import matplotlib.pyplot as plt
import pytesseract

In [ ]:
IN_DIR = "../data/raw"
IN_FILE = "sample.png"
IN_PATH = os.path.join(IN_DIR, IN_FILE)
CONFIG_DIR = "../configs"
JSON_FILE = "template.json"
CONFIG_PATH = os.path.join(CONFIG_DIR, JSON_FILE)

In [ ]:
img = cv2.imread(IN_PATH)
cfg = json.load(open(CONFIG_PATH))

In [ ]:
def show(im, figsize=(4, 2)):
    plt.figure(figsize=figsize)
    cmap = "gray" if im.ndim == 2 else None
    disp = im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    plt.imshow(disp, cmap=cmap)
    plt.axis("off")
    plt.show()

In [ ]:
top = cfg["teams"]["blue"]["row_tops"][0]
h = cfg["row_h"]
x1, x2 = cfg["col_x"]["DMG"]
cell = img[top:top+h, x1:x2]

In [ ]:
show(cell)

In [ ]:
def preprocess(cell, scale=3):
    resize = cv2.resize(cell, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(resize, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.mean(thr) < 127:
        thr = cv2.bitwise_not(thr)
    return thr

In [ ]:
proc = preprocess(cell)
show(proc)

In [ ]:
def read_number(cell):
    proc = preprocess(cell)
    c = "--psm 7 -c tessedit_char_whitelist=0123456789,"
    return pytesseract.image_to_string(proc, config=c).strip().replace(',', '')

In [ ]:
print("DMG reads as:", read_number(cell))

In [ ]:
value = read_number(cell) # Note: returns as string value, cast to int
assert int(value) == 3919, f"Expected 3919, but got {value}"

In [ ]:
nx1, nx2 = cfg["name_x"]
name_cell = img[top:top+h, nx1:nx2]
show(name_cell, figsize=(6,2))

In [ ]:
name = preprocess(name_cell) # Note: returns as numpy.ndarray
print("tesseract:", pytesseract.image_to_string(name, config="--psm 7").strip())

In [ ]:
print(name, type(name))

In [ ]:
assert np.all(name == "0 LUCKYOW!"), f"Expected all LUCKYOWL, but got {name}"

Names: EasyOCR

In [ ]:
import easyocr

In [ ]:
reader = easyocr.Reader(["en"], gpu=True)
res = reader.readtext(name_cell, detail=0)
print("EasyOCR:", " ".join(res))

In [ ]:
print(res, type(res))

In [ ]:
assert res[0] == "LUCKYOWL", f"Expected LUCKYOWL, but got {res}"